# M25 · Contextual Bandits — Toy Example, Step by Tiny Step

**Companion to lesson M25.** Learn which of 3 arms pays best from feedback alone, using **epsilon-greedy** (mostly exploit the best-so-far, occasionally explore), and see the **break case** where zero exploration gets stuck on a bad arm.

## Step 0 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 4)

def log(label, value):
    print(f"[{label}] {value}")

log("setup", "tools ready — seed fixed to 0")

## Step 1 · Epsilon-greedy learns the best arm

Three arms pay 1 with hidden probabilities 0.2, 0.5, 0.8. We keep a running average reward `Q` per arm. Each round: with prob `eps` explore a random arm, else pull the current best; then update that arm's average.

In [ ]:
true_p = [0.2, 0.5, 0.8]; Q = [0.0, 0.0, 0.0]; N = [0, 0, 0]; eps = 0.1
rewards = []
for t in range(500):
    arm = np.random.randint(3) if np.random.rand() < eps else int(np.argmax(Q))
    reward = 1 if np.random.rand() < true_p[arm] else 0
    N[arm] += 1; Q[arm] += (reward - Q[arm]) / N[arm]        # incremental average
    rewards.append(reward)
log("learned Q (est. reward per arm)", np.round(Q, 2).tolist())
log("pulls per arm", N)
assert int(np.argmax(Q)) == 2                                 # it found the 0.8 arm

plt.plot(np.cumsum(rewards) / (np.arange(500) + 1))
plt.title("epsilon-greedy: average reward climbs"); plt.xlabel("round"); plt.ylabel("avg reward"); plt.show()

▶ What you'll see: Q[arm2] ≈ 0.8, most pulls on arm 2, and average reward rising toward 0.8.

## Step 2 · Break case: no exploration (eps=0) gets stuck

With `eps=0` the agent only ever pulls whatever looked best first. One lucky early pull on a **bad** arm can lock it in forever — it never tries the truly-best arm.

In [ ]:
Q0 = [0.0, 0.0, 0.0]; N0 = [0, 0, 0]
Q0[0] = 1.0; N0[0] = 1                                        # a lucky first win on the WORST arm (0.2)
for t in range(500):
    arm = int(np.argmax(Q0))                                 # pure greedy, no exploration
    reward = 1 if np.random.rand() < true_p[arm] else 0
    N0[arm] += 1; Q0[arm] += (reward - Q0[arm]) / N0[arm]
log("greedy-only pulls per arm", N0)
assert N0[2] == 0                                            # it NEVER even tried the best arm
print("Lesson: without exploration you can lock onto a bad arm forever.")

▶ What you'll see: all pulls stuck on arm 0, zero pulls on the truly-best arm 2.

## Recap

- A bandit learns from **reward feedback only** — no labels.
- **Epsilon-greedy** balances exploiting the best-so-far with exploring alternatives.
- **No exploration** risks locking onto a bad arm; some exploration is essential.